# Is ReLU² why FP4 training works? — the TPU runs (JAX / Flax NNX / Qwix)

This notebook trains the study's small GPT on a Colab TPU. It uses the same model, optimizer and tokens as the Mac MLX runs (`mlx_mac/`), and crosses **7 MLP activations** with **8 number-format recipes**. Repo: [github.com/atveit/relu_squared_fp4](https://github.com/atveit/relu_squared_fp4).

**Pick the runtime:** *Runtime → Change runtime type → TPU v5e-1* (or *TPU v6e-1*).
- Each chip needs its own pass of the notebook.
- One results file holds both passes. Every record stores the chip in its `device` field (`TPU v5 lite` = v5e, `TPU v6 lite` = v6e), and the grid skips only runs already done **on the current chip**.

**What is real and what is simulated:**
- **int8 is real.** Qwix quantized training runs int8 × int8 → int32 GEMMs on the TPU's MXU, with stochastic rounding on the gradients.
- **FP8 and FP4 are simulated.** TPU v5e and v6e have no FP4 hardware, and no FP8 either. The `sim_*` recipes fake-quantize with the Mac simulator's exact numerics (`mlx_mac/lowp.py`): fp32 GEMMs on values rounded to the format.
- **Two baselines:**
  - `fp32` is the Mac's baseline; compare it with the `sim_*` arms.
  - `bf16` is the TPU-native baseline; compare it with real `int8`.

**LR schedule:** the grids use the **intended** schedule from `common/config.py`: constant LR, then a linear decay over the last 40% of steps to 0.1×.
- The Mac grid in `mlx_mac/results/lowp_A.jsonl` was run before a fix (2026-10-31). MLX's uint64 step counter wrapped around in the schedule, so the LR sat at 0.1× for the first 60% of each run (see `jax_tpu/LEARNINGS.md`).
- Optional step 5b reruns a few arms with that old schedule replicated (`--schedule mlx_uint64_bug`) and compares them with the Mac rows.
- Every record stores its `schedule`.

**Steps** (run the cells top to bottom):
1. Check the runtime.
2. Install the pinned packages.
3. Get the code (`git clone`; fallbacks: upload a zip, or Google Drive).
4. Mount Drive. Optional, but it keeps results safe if Colab disconnects.
5. Run the 20-step smoke run. It prints step times and an estimate for the full grid on your chip.
   - Step 5b (optional): cross-check 12 runs against the Mac grid, about 20–40 min.
6. Run the **small grid**: 7 activations × 8 recipes × 2 seeds = 112 runs of 600 steps.
   - It runs the core arms first (ReLU², LeakyReLU², SwiGLU × fp32 / bf16 / int8 / sim_int8 / sim_nvfp4).
   - It is resumable: re-run the cell after a disconnect and it skips finished runs.
7. Save the results to Drive and download them.
8. (Optional) Run the GPT-2-small **scale** grid.

**Expected runtime:** the step-5 cell prints a measured estimate. As a rough guide:
- small grid: about 3–5 h on v5e-1, about 1.5–3 h on v6e-1;
- scale grid: about 10+ h on v5e.

**Sending the results back:** download `results/jax_small.jsonl` (plus `jax_tpu20.jsonl`, and `jax_small_errors.log` if it exists) and put them in `results/` of the repo on the Mac. If the files are on Drive, you can also give the Drive folder path instead.

## 1. Runtime check
Runs in a subprocess, so the kernel never imports JAX and the install in step 2 needs no restart.

In [ ]:
import os, subprocess
print("TPU_ACCELERATOR_TYPE =", os.environ.get("TPU_ACCELERATOR_TYPE"), "| COLAB_TPU_ADDR =", os.environ.get("COLAB_TPU_ADDR"))
!python -c "import jax; d = jax.devices(); print('jax', jax.__version__, '| backend', jax.default_backend(), '|', len(d), 'device(s):', d[0].device_kind); print(d)"
!python --version

## 2. Install the pinned packages
- These versions passed the CPU smoke test and the MLX parity checks on the Mac: `flax 0.12.12`, `optax 0.2.8`, `qwix 0.1.8` (with `jax 0.12.0.1`).
- The cell keeps Colab's preinstalled `jax` / `jaxlib` / `libtpu`. Reinstalling them can break the TPU runtime, so they are passed to pip as fixed versions.
- Only if the pins can't resolve against them does the cell install the tested stack, `jax[tpu]==0.12.0.1`, which brings its own matching `libtpu`. The runtime then needs a restart.

In [ ]:
# Keep Colab's preinstalled jax / jaxlib / libtpu: pip gets them as fixed versions, so it can't upgrade them.
import subprocess, sys
from importlib.metadata import version, PackageNotFoundError

def v(pkg):
    try:
        return version(pkg)
    except PackageNotFoundError:
        return None

print("preinstalled: jax", v("jax"), "| jaxlib", v("jaxlib"), "| libtpu", v("libtpu") or v("libtpu-nightly"))
PINS = ["flax==0.12.12", "optax==0.2.8", "qwix==0.1.8"]
KEEP = [f"jax=={v('jax')}", f"jaxlib=={v('jaxlib')}"] if v("jax") and v("jaxlib") else []
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS, *KEEP])
if r.returncode != 0 or not KEEP:
    print("the pins don't resolve against the preinstalled jax: installing the tested stack jax[tpu]==0.12.0.1 + libtpu")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jax[tpu]==0.12.0.1", *PINS], check=True)
    print(">>> Restart the runtime (Runtime -> Restart session), then continue from step 3. <<<")
!python -c "import jax, flax, optax, qwix; from importlib.metadata import version as v; print('jax', jax.__version__, '| flax', v('flax'), '| optax', v('optax'), '| qwix', v('qwix')); print(jax.devices())"

## 3. Get the code
Default: clone the public repo. If the clone fails (for example, the repo is private), use one of the two fallback cells below it.

In [ ]:
import os
REPO = "/content/relu_squared_fp4"
if os.path.isdir(f"{REPO}/.git"):
    !git -C {REPO} pull --ff-only
elif not os.path.isdir(f"{REPO}/jax_tpu"):
    !git clone https://github.com/atveit/relu_squared_fp4 {REPO}
%cd {REPO}
!git log --oneline -1 2>/dev/null; ls

In [ ]:
# Fallback A: upload colab_bundle.zip (common/, jax_tpu/, __init__.py) from the Mac.
# from google.colab import files
# import os, zipfile
# up = files.upload()                                   # pick colab_bundle.zip
# os.makedirs("/content/relu_squared_fp4/results", exist_ok=True)
# zipfile.ZipFile(next(iter(up))).extractall("/content/relu_squared_fp4")
# %cd /content/relu_squared_fp4

In [ ]:
# Fallback B: the bundle on Google Drive (upload colab_bundle.zip to MyDrive/relu_squared_fp4/ first).
# from google.colab import drive
# drive.mount("/content/drive")
# !mkdir -p /content/relu_squared_fp4/results && unzip -o -q /content/drive/MyDrive/relu_squared_fp4/colab_bundle.zip -d /content/relu_squared_fp4
# %cd /content/relu_squared_fp4

## 4. Mount Google Drive (recommended)
- The grid writes into the clone, at `results/jax_small.jsonl`.
- After every chunk, the grid cell copies the file to `MyDrive/relu_squared_fp4_results/`.
- In a new session, an existing Drive copy is first copied back into the clone, so the grid resumes where it stopped.
- To skip Drive, set `USE_DRIVE = False`. Results are then only in `/content`, which is lost when the runtime is recycled.

In [ ]:
import os, shutil
USE_DRIVE = True
DRIVE_DIR = "/content/drive/MyDrive/relu_squared_fp4_results"
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs("results", exist_ok=True)

def _lines(p):
    return sum(1 for _ in open(p)) if os.path.exists(p) else 0

def restore(name):
    # Drive -> clone, when the Drive copy has more records (a new session resuming a grid).
    if USE_DRIVE and _lines(f"{DRIVE_DIR}/{name}") > _lines(f"results/{name}"):
        shutil.copy(f"{DRIVE_DIR}/{name}", f"results/{name}")
        print(f"restored results/{name} from Drive ({_lines(f'results/{name}')} records)")

def sync(*names):
    # clone -> Drive (results and error logs)
    if not USE_DRIVE:
        return
    for name in names:
        for p in (f"results/{name}", f"results/{name.replace('.jsonl', '_errors.log')}"):
            if os.path.exists(p):
                shutil.copy(p, f"{DRIVE_DIR}/{os.path.basename(p)}")
    print("synced to", DRIVE_DIR, ":", sorted(os.listdir(DRIVE_DIR)))

for name in ("jax_tpu20.jsonl", "jax_xcheck.jsonl", "jax_small.jsonl", "jax_scale.jsonl"):
    restore(name)

## 5. 20-step smoke run, and the runtime estimate
- What it runs: ReLU² × {fp32, bf16, int8 (Qwix), sim_nvfp4}, 20 steps each, with the full 32 × 512-token batch.
- First run only: it also downloads the two FineWeb shards (~400 MB) into `data/`.
- Expect a loss of ≈ 10.83 (= ln 50304) at step 0, falling by step 19.
- `first` is the compile time.
- If a run says *out of memory*, it retries with micro-batches (`--accum`, same math).

In [ ]:
!python jax_tpu/train_jax.py --grid tpu20 --out results/jax_tpu20.jsonl --log-every 5 --verbose
sync("jax_tpu20.jsonl")

In [ ]:
import json
recs = [json.loads(l) for l in open("results/jax_tpu20.jsonl")]
dev = recs[-1]["device"]
recs = {r["recipe"]: r for r in recs if r["device"] == dev}
print(f"device: {dev}\n{'recipe':<10} {'step ms':>8} {'compile s':>9} {'eval s':>7} {'accum':>5}  losses")
for k, r in recs.items():
    print(f"{k:<10} {r['step_ms']:>8.0f} {r['first_step_s']:>9.1f} {r['eval_s']:>7.1f} {r['accum']:>5}  "
          f"{[round(l, 3) for _, l in r['losses']]}")
# Small grid = per (activation, seed): fp32, bf16, int8, and five sim_* recipes (the sim_nvfp4 time stands in for all
# five). Eval in the grid is 8x the tokens of this smoke run; seed 2 reuses seed 1's compile from the cache.
kind = lambda rec: rec if rec in recs else "sim_nvfp4"
small = ["fp32", "bf16", "int8", "sim_fp8", "sim_int8", "sim_nvfp4", "sim_nvfp4h", "sim_mxfp4"]
per_run = {k: 600 * recs[kind(k)]["step_ms"] / 1000 + 8 * recs[kind(k)]["eval_s"] for k in small}
compile_s = {k: recs[kind(k)]["first_step_s"] for k in small}
total = sum(7 * 2 * per_run[k] + 7 * compile_s[k] for k in small)
print(f"\nestimated small grid on {dev}: {total / 3600:.1f} h for 112 runs "
      f"(core chunk of 30 runs: ~{total * 30 / 112 / 3600:.1f} h); packed activations are ~5-10% slower")

## 5b. (Optional) Cross-check with the Mac grid as it ran
- **Arms:** ReLU² and LeakyReLU² × fp32 / bf16 / sim_nvfp4 × seeds 1–2: 12 runs of 600 steps, with the MLX LR-schedule bug replicated (`--grid xcheck`, `--schedule mlx_uint64_bug`).
- **Comparison:** the Mac rows in `mlx_mac/results/lowp_A.jsonl`.
  - Initializations differ across frameworks, so expect agreement within seed noise: a few mnat (the Mac's seed 1 vs seed 2 differ by ~1–3 mnat).
  - fp32 is the like-for-like arm. bf16 also differs by its bf16 GEMMs.
- On the Mac CPU, exact weight-transfer tests already matched MLX step for step, for every activation and every sim recipe. This cell checks the TPU numerics end to end.

In [ ]:
restore("jax_xcheck.jsonl")
!python jax_tpu/train_jax.py --grid xcheck --out results/jax_xcheck.jsonl
sync("jax_xcheck.jsonl")
import json
mac = {}
for l in open("mlx_mac/results/lowp_A.jsonl"):
    r = json.loads(l)
    mac[(r["arm"], r["seed"], r["n"])] = r["val_loss"]
arm = {("relu2", "fp32"): "relu2_metal", ("relu2", "sim_nvfp4"): "relu2_metal+q=nvfp4",
       ("lrelu2", "fp32"): "lrelu2", ("lrelu2", "sim_nvfp4"): "lrelu2+q=nvfp4",
       ("relu2", "bf16"): "relu2_metal", ("lrelu2", "bf16"): "lrelu2"}
print(f"{'act':<8} {'recipe':<10} {'seed':>4} {'TPU val':>9} {'Mac val':>9} {'TPU-Mac mnat':>13}")
for l in open("results/jax_xcheck.jsonl"):
    r = json.loads(l)
    m = mac.get((arm[(r["act"], r["recipe"])], r["seed"], r["n"]))
    d = f"{1000 * (r['val_loss'] - m):+13.1f}" if m is not None else f"{'-':>13}"
    print(f"{r['act']:<8} {r['recipe']:<10} {r['seed']:>4} {r['val_loss']:>9.4f} {m if m is not None else float('nan'):>9.4f} {d}   [{r['device']}]")

## 6. The small grid (112 runs, resumable)
- Config: tiny model (d = 256, 4 layers, 29M parameters), 600 steps, seeds 1 and 2, intended LR schedule.
- Order:
  - **Core chunk first**: ReLU², LeakyReLU², SwiGLU × fp32 / bf16 / int8 / sim_int8 / sim_nvfp4 (30 runs).
  - **Then the rest**, one recipe at a time.
- After each chunk, the results file is copied to Drive.
- After a disconnect: re-run steps 2, 3 and 4, then this cell. Finished runs are skipped.
- A failed run is logged to `results/jax_small_errors.log`, and the grid continues.

*Parity (2026-10-31): the JAX trainer matches the MLX trainer step for step from identical weights, for every activation and every simulated format (`jax_tpu/LEARNINGS.md`).*

In [ ]:
CORE_ACTS = "relu2,lrelu2,swiglu"
CORE_RECIPES = "fp32,bf16,int8,sim_int8,sim_nvfp4"
ALL_RECIPES = ["fp32", "bf16", "int8", "sim_int8", "sim_nvfp4", "sim_nvfp4h", "sim_mxfp4", "sim_fp8"]

restore("jax_small.jsonl")
!python jax_tpu/train_jax.py --grid small --acts {CORE_ACTS} --recipes {CORE_RECIPES}
sync("jax_small.jsonl")
for r in ALL_RECIPES:   # the remaining activations (and any core gaps), one recipe per chunk
    !python jax_tpu/train_jax.py --grid small --recipes {r}
    sync("jax_small.jsonl")

### Quick look at the results so far
The paired penalty is val(recipe) − val(baseline) in milli-nats, for the same activation and seed. sim_* arms are compared with fp32, int8 with bf16.

In [ ]:
import json, collections, statistics as st
rows = [json.loads(l) for l in open("results/jax_small.jsonl")]
dev = rows[-1]["device"]
v = {(r["act"], r["recipe"], r["seed"]): r["val_loss"] for r in rows if r["device"] == dev and r["n"] == 600 and r.get("schedule") == "intended"}
acts = ["relu2", "lrelu2", "swiglu", "xielu", "s2glu_r2", "eglu_r2n", "eml"]
recipes = ["fp32", "bf16", "int8", "sim_fp8", "sim_int8", "sim_mxfp4", "sim_nvfp4", "sim_nvfp4h"]
base = lambda rec: "bf16" if rec in ("int8", "bf16") else "fp32"
print(f"{dev}: {len(v)} runs\nval loss (mean over seeds)")
print(f"{'act':<10}" + "".join(f"{r:>11}" for r in recipes))
for a in acts:
    cells = [st.mean(x) if (x := [v[(a, r, s)] for s in (1, 2) if (a, r, s) in v]) else None for r in recipes]
    print(f"{a:<10}" + "".join(f"{c:>11.4f}" if c is not None else f"{'-':>11}" for c in cells))
print("\nprecision penalty vs the paired baseline [mnat]")
print(f"{'act':<10}" + "".join(f"{r:>11}" for r in recipes if r not in ("fp32", "bf16")))
for a in acts:
    cells = []
    for r in recipes:
        if r in ("fp32", "bf16"):
            continue
        d = [1000 * (v[(a, r, s)] - v[(a, base(r), s)]) for s in (1, 2) if (a, r, s) in v and (a, base(r), s) in v]
        cells.append(f"{st.mean(d):>+11.1f}" if d else f"{'-':>11}")
    print(f"{a:<10}" + "".join(cells))

## 7. Save and download the results
Put the downloaded files in `results/` of the repo on the Mac.

In [ ]:
sync("jax_tpu20.jsonl", "jax_xcheck.jsonl", "jax_small.jsonl", "jax_scale.jsonl")
from google.colab import files
import os
for p in ("results/jax_small.jsonl", "results/jax_tpu20.jsonl", "results/jax_xcheck.jsonl", "results/jax_small_errors.log",
          "results/jax_scale.jsonl", "results/jax_scale_errors.log"):
    if os.path.exists(p):
        files.download(p)

## 8. (Optional) The scale grid: GPT-2-small shape
- Setup: d = 768, 12 layers, 1024 context, 32 sequences per step, 2,000 steps.
- Arms: ReLU², LeakyReLU², SwiGLU, the bounded EML gate × bf16 / int8 / fp32 / sim_nvfp4 / sim_nvfp4h, seed 1 (20 runs).
- Memory: per-block rematerialization is on, and micro-batches are chosen from the chip's memory.
- Time: hours per run on a v5e-1 (more for the fp32 / sim arms). Run it only with time to spare, preferably on v6e.
- Resumable, like the small grid.

In [ ]:
restore("jax_scale.jsonl")
for r in ["bf16", "int8", "sim_nvfp4", "fp32", "sim_nvfp4h"]:
    !python jax_tpu/train_jax.py --grid scale --recipes {r} --log-every 100
    sync("jax_scale.jsonl")